# Table Types in Snowflake:

- Permanent
- Transient
- Temporary
- Dynamic
- Iceberg (Covered in a dedicated Lab)

In [ ]:
%%sql -r dataframe_1
SELECT CURRENT_ROLE()

In [ ]:
%%sql -r dataframe_2
DROP DATABASE IF EXISTS DEMO_DB;

## Step 1: Setup

In [ ]:
%%sql -r dataframe_3
-- Create new database and schema

CREATE DATABASE IF NOT EXISTS DEMO_DB;
CREATE SCHEMA IF NOT EXISTS DEMO_SCHEMA;
USE DATABASE DEMO_DB;
USE SCHEMA DEMO_SCHEMA;

In [ ]:
%%sql -r dataframe_4
CREATE WAREHOUSE IF NOT EXISTS DEMO_WH WITH WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 300;
USE WAREHOUSE DEMO_WH;

## Step 2: Create Tables 

In [ ]:
%%sql -r dataframe_5
-- Create one table of each type to observe their differences.
-- Create four tables with the same structure but different types:

-- Regular (Permanent) Table
CREATE TABLE regular_table (
    id INT,
    name STRING,
    created_at TIMESTAMP
);

-- Transient Table
CREATE TRANSIENT TABLE transient_table (
    id INT,
    name STRING,
    created_at TIMESTAMP
);

-- Temporary Table
CREATE TEMPORARY TABLE temp_table (
    id INT,
    name STRING,
    created_at TIMESTAMP
);

-- Dynamic Table (requires a source table or query)
-- First, create a source table for the dynamic table
CREATE TABLE source_table (
    id INT,
    name STRING,
    created_at TIMESTAMP
);

-- Create Dynamic Table based on source_table
CREATE DYNAMIC TABLE dynamic_table
    TARGET_LAG = '1 hour'
    WAREHOUSE = DEMO_WH
    AS
    SELECT id, name, created_at
    FROM source_table
    WHERE created_at >= CURRENT_DATE;

In [ ]:
%%sql -r dataframe_6
SELECT current_warehouse()

## Step 3: Insert and Query Data

In [ ]:
%%sql -r dataframe_7
-- Insert into regular table
INSERT INTO regular_table VALUES
    (1, 'Alice', CURRENT_TIMESTAMP),
    (2, 'Bob', CURRENT_TIMESTAMP);

-- Insert into transient table
INSERT INTO transient_table VALUES
    (1, 'Alice', CURRENT_TIMESTAMP),
    (2, 'Bob', CURRENT_TIMESTAMP);

-- Insert into temporary table
INSERT INTO temp_table VALUES
    (1, 'Alice', CURRENT_TIMESTAMP),
    (2, 'Bob', CURRENT_TIMESTAMP);

-- Insert into source table (for dynamic table)
INSERT INTO source_table VALUES
    (1, 'Alice', CURRENT_TIMESTAMP),
    (2, 'Bob', CURRENT_TIMESTAMP);

-- Note! We do not insert into Dynamic Tables

In [ ]:
%%sql -r dataframe_8
SELECT * FROM regular_table;


In [ ]:
%%sql -r dataframe_9
SELECT * FROM transient_table;


In [ ]:
%%sql -r dataframe_10
SELECT * FROM temp_table;


In [ ]:
%%sql -r dataframe_11
SELECT * FROM dynamic_table;

## Step 4: Explore Retention and Storage

In [ ]:
%%sql -r dataframe_12
-- Get last query id
SET id = last_query_id();


In [ ]:
%%sql -r dataframe_13
UPDATE regular_table SET name = 'Charlie' WHERE id = 1;
UPDATE transient_table SET name = 'Charlie' WHERE id = 1;
UPDATE temp_table SET name = 'Charlie' WHERE id = 1;

In [ ]:
%%sql -r dataframe_14
-- Time Travel for regular table
SELECT * FROM regular_table BEFORE (STATEMENT => $id);


In [ ]:
%%sql -r dataframe_15
-- Time Travel for transient table
SELECT * FROM transient_table BEFORE (STATEMENT => $id);

In [ ]:
%%sql -r dataframe_16
-- Time Travel for temporary table
SELECT * FROM temp_table BEFORE (STATEMENT => $id);

In [ ]:
-- Start a new notebook or worksheet and check these:

use role accountadmin;
use schema DEMO_DB.DEMO_SCHEMA;  

SELECT * FROM regular_table;
SELECT * FROM transient_table;
SELECT * FROM temp_table;
SELECT * FROM dynamic_table;

/* Expected Outcome:

Regular and Transient Tables: Data persists across sessions.
Temporary Table: Table no longer exists in the new session (query fails with "Table does not exist").
Dynamic Table: Data persists and reflects the latest refresh based on the source table.

*/

## Step 5: Dynamic Table Specifics

In [ ]:
%%sql -r dataframe_18
-- Insert new data into the source table:

INSERT INTO source_table VALUES
    (3, 'Charlie', CURRENT_TIMESTAMP);

In [ ]:
%%sql -r dataframe_19
-- Manually trigger a refresh for the dynamic table (if needed):

ALTER DYNAMIC TABLE dynamic_table REFRESH;

In [ ]:
%%sql -r dataframe_20
-- Query the dynamic table to see updated data:

SELECT * FROM dynamic_table;

In [ ]:
%%sql -r dataframe_21
-- Check the refresh history:

SELECT * FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY())
WHERE NAME = 'DYNAMIC_TABLE';

## Step 6: Cleanup

In [ ]:
%%sql -r dataframe_22
DROP DATABASE IF EXISTS DEMO_DB;
DROP WAREHOUSE IF EXISTS DEMO_WH;
USE WAREHOUSE COMPUTE_WH;